In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Render safely without opening native GUI windows.
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize


In [2]:
NORMALIZATOR = 1e-10

In [3]:
def generate_fishing_grounds(n_grounds = 10, x_max = 100, y_max = 100):
    return np.random.rand(n_grounds, 2) * np.array([x_max, y_max])

In [4]:
def generate_ships(n_ships = 10, x_max = 100, y_max = 100):
    return np.random.rand(n_ships, 2) * np.array([x_max, y_max])

In [5]:
def generate_ships_at_harbor(h, n_ships = 10):
    """
    Returns an n_ships x 2 array of ships at the harbor h with gaussian noise
    """
    # return np.array([h + np.random.normal(0, 1, 2) for _ in range(n_ships)])
    return np.array([h for _ in range(n_ships)])

In [6]:
def calculate_ship_distance_from_grounds(ships, grounds):
    return np.linalg.norm(ships[:, np.newaxis] - grounds, axis=2)

In [7]:
def d_matrix(ships, grounds):
    # Vectorized difference calculation
    differences = grounds - ships[:, np.newaxis]
    distances = np.linalg.norm(differences, axis=2, keepdims=True)
    return differences / (distances + NORMALIZATOR)

In [8]:
def ship_ground_gaussians(ships, grounds, alpha, b=1, c=0):
    distances = np.linalg.norm(ships[:, np.newaxis] - grounds, axis=2)
    return b*np.exp(-alpha * distances**2) + c

In [9]:
def calculate_ship_distance_from_ships(ships):
    return np.linalg.norm(ships[:, np.newaxis] - ships, axis=2)

In [10]:
def harbor_attraction(U, ships, h):
    diff = h - ships
    diff_norm = np.linalg.norm(diff, axis=1, keepdims=True)
    return U[:, -2, np.newaxis] * (diff / (diff_norm + NORMALIZATOR))

In [11]:
def attraction_matrix(U, G, D, F):
    F_reshaped = F.reshape(1, -1, 1)
    return U[:, :-2, None] * G[:, :, None] * D * F_reshaped

In [12]:
def calculate_F_prime(t, F, C, rate):
    C_sum = np.sum(C, axis=0)
    return -rate * F * C_sum

In [13]:
def calculate_S_matrix(A, H):
    return np.sum(A, axis=1).reshape(-1, 2) + H

In [14]:
def calculate_x_prime(U, S):

    # Reshape U[:,-2] to have an extra dimension
    U_reshaped = U[:,-1].reshape(-1, 1)

    return U_reshaped * S

Take orthogonal unit vector to s_i, namely r_i

inner product that with p_i, take abs value

find minimum of that <r_i, p_i>

Penalize ||u_i,j(t-1) - u_i,j(t)||

so minimize over u_ij for all i, j <r_i, p_i> + ||u_i,j(t-1) - u_i,j(t)||

In [ ]:
# Costate derivatives are implemented together in costate() below.


In [ ]:
# Costate derivatives are implemented together in costate() below.


In [ ]:
# Costate derivatives are implemented together in costate() below.


In [ ]:
# Costate derivatives are implemented together in costate() below.


In [ ]:
def state(t, y, params):
    U, alpha, sigma, beta, rate, grounds, h, x_size, F_size, x_shape= params
    b = 1
    x = y[:x_size].reshape(x_shape)  # reshape x back to its original shape
    F = y[x_size:]

    differences = grounds - x[:, np.newaxis]
    distances = np.linalg.norm(differences, axis=2, keepdims=True)

    # ship_distances = np.linalg.norm(ship_diffences, axis=2)

    # Calculate D matrix
    D = differences / (distances + NORMALIZATOR)

    # Calculate G matrix
    G = ship_ground_gaussians(x, grounds, alpha, b=b)

    # Calculate Collection matrix
    C = ship_ground_gaussians(x, grounds, sigma, b=b)

    # Calculate A matrix
    A = attraction_matrix(U, G, D, F)

    # Calculate H
    # H = harbor_attraction(U, x, h)


    S = np.sum(A, axis=1).reshape(-1, 2)

    # S = S + H

    S_normal = S / (np.linalg.norm(S, axis=1, keepdims=True) + NORMALIZATOR)

    F_prime = calculate_F_prime(t, F, C, rate)

    x_prime = calculate_x_prime(U, S_normal)

    return np.concatenate((x_prime.flatten(), F_prime.flatten()))

def costate(t, y, params):
    # Differentiate the IMPLEMENTED forward dynamics, including NORMALIZATOR.
    x, F, U, alpha, sigma, beta, epsilon, rate, gamma, grounds, h, p_size, pf_size, p_shape = params
    x = x.reshape(p_shape)
    p, pf = y[:p_size].reshape(p_shape), y[p_size:]
    q = grounds - x[:, None]
    rho = np.linalg.norm(q, axis=2)
    e = NORMALIZATOR
    D = q / (rho[..., None] + e)
    G = np.exp(-alpha * rho**2)
    C = np.exp(-sigma * rho**2)
    S = np.sum((U[:, :-2] * G * F)[..., None] * D, axis=1)
    sn = np.linalg.norm(S, axis=1)
    # At exact zero the arclength term is nondifferentiable; choose zero
    # subgradient. Such a point is not a certified smooth PMP solution.
    safe_rho = np.where(rho > 0, rho, 1.)
    safe_sn = np.where(sn > 0, sn, 1.)
    Q = np.eye(2)[None] / (sn + e)[:, None, None] - (
        S[:, :, None] * S[:, None, :]
        / (safe_sn * (sn + e)**2)[:, None, None])
    Dxs = np.sum((U[:, :-2] * G * F / (rho + e))[..., None, None] * (
        (2*alpha + 1/(safe_rho*(rho+e)))[..., None, None]
        * q[..., :, None] * q[..., None, :] - np.eye(2)), axis=1)
    # H_s includes the derivative of gamma * |x_dot|, since |x_dot| != v
    # for the notebook's regularized normalization.
    Hs = U[:, -1, None] * (np.einsum('nij,nj->ni', Q, p)
          - gamma * e * S / (safe_sn * (sn+e)**2)[:, None])
    diff = x[None] - x[:, None]  # x_l - x_i, with the coordinate axis last
    crowd = 4*beta*np.sum(diff / (
        epsilon + np.sum(diff**2, axis=2))[..., None]**2, axis=1)
    fish = 2*rate*sigma*np.sum((pf*F*C)[..., None]*q, axis=1)
    p_prime = crowd + fish - np.einsum('nji,nj->ni', Dxs, Hs)
    pf_prime = rate*pf*C.sum(axis=0) - np.sum(
        U[:, :-2]*G*np.einsum('nkj,nj->nk', D, Hs), axis=0)
    return np.r_[p_prime.ravel(), pf_prime]

def control_intervals(U_tensor, t_final):
    """Exact intervals of the notebook's nearest-time-node control convention."""
    dt = t_final/(len(U_tensor)-1)
    boundaries = np.r_[0., (np.arange(len(U_tensor)-1)+.5)*dt, t_final]
    changes = np.r_[0, np.flatnonzero(np.any(U_tensor[1:] != U_tensor[:-1],axis=(1,2)))+1, len(U_tensor)]
    return [(boundaries[a],boundaries[b],U_tensor[a]) for a,b in zip(changes[:-1],changes[1:])]


def piecewise_result(pieces, times):
    from types import SimpleNamespace
    pieces = sorted(pieces,key=lambda entry:entry[0])
    starts = np.array([entry[0] for entry in pieces])
    def dense(t):
        points = np.atleast_1d(t)
        idx = np.clip(np.searchsorted(starts,points,side='right')-1,0,len(pieces)-1)
        values = np.column_stack([pieces[k][1](point) for point,k in zip(points,idx)])
        return values[:,0] if np.ndim(t)==0 else values
    return SimpleNamespace(t=times,y=dense(times),sol=dense,success=True,message='Completed all control intervals.')


def solve_state_equations(state, U_tensor, params, t_final, y0, num_timesteps):
    """Forward solve and actual-cost quadrature, split at EVERY control switch.

    Append off-diagonal crowding and distance costs. beta*N*T/epsilon is an
    omitted constant. Tiny denominator regularizers remain unchanged.
    """
    import time
    deadline = time.monotonic()+20.
    alpha,sigma,beta,rate,grounds,h,x_size,F_size,x_shape,epsilon,gamma = params
    z = np.r_[y0,0.,0.]
    pieces = []
    for start,end,U in control_intervals(U_tensor,t_final):
        def augmented(t,z):
            if time.monotonic()>deadline:
                raise RuntimeError('Forward solve exceeded 20 seconds; stiff trajectory.')
            derivative = state(t,z[:-2],[U]+params[:9])
            x = z[:x_size].reshape(x_shape)
            inv = 1/(epsilon+np.sum((x[:,None]-x[None])**2,axis=2))
            np.fill_diagonal(inv,0.)
            distance = gamma*np.linalg.norm(derivative[:x_size].reshape(x_shape),axis=1).sum()
            return np.r_[derivative,beta*inv.sum(),distance]
        sol = solve_ivp(augmented,[start,end],z,dense_output=True,
                        max_step=1.,rtol=1e-6,atol=1e-8)
        if not sol.success:
            raise RuntimeError(sol.message)
        pieces.append((start,sol.sol))
        z = sol.y[:,-1]
    return piecewise_result(pieces,np.linspace(0,t_final,num_timesteps))


def solve_costate_equations(costate,state_sol,U_tensor,params_costate,
                            t_final,p0,x_size,F_size,num_timesteps):
    """Backward solve at the SAME physical time, split at control switches."""
    import time
    deadline = time.monotonic()+20.
    linear_params = params_costate.copy()
    linear_params[2] = 0.  # remove affine beta source
    linear_params[5] = 0.  # remove affine gamma source
    basis = np.eye(len(p0))
    p = p0.copy()
    pieces = []
    for start,end,U in reversed(control_intervals(U_tensor,t_final)):
        def parameters(t,model_params):
            if time.monotonic()>deadline:
                raise RuntimeError('Adjoint solve exceeded 20 seconds; no convergence claim.')
            z = state_sol.sol(t)
            return [z[:x_size],z[x_size:x_size+F_size],U]+model_params
        def backward(t,p):
            return costate(t,p,parameters(t,params_costate))
        def jacobian(t,p):
            # Exact affine-in-p Jacobian; no finite-difference overflow.
            params = parameters(t,linear_params)
            return np.column_stack([costate(t,column,params) for column in basis])
        sol = solve_ivp(backward,[end,start],p,jac=jacobian,method='BDF',
                        dense_output=True,max_step=1.,rtol=1e-5,atol=1e-7)
        if not sol.success:
            raise RuntimeError(sol.message)
        pieces.append((start,sol.sol))
        p = sol.y[:,-1]
    return piecewise_result(pieces,np.linspace(t_final,0,num_timesteps))


def fishing_objective(sol,x_size,F_size,delta):
    """J minus its control-independent self-crowding constant."""
    return float(sol.y[-2,-1]+sol.y[-1,-1]+delta*sol.y[x_size:x_size+F_size,-1].sum())


In [20]:
# def state(t, y, params):
#     alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape= params
#     b = 1
#     x = y[:x_size].reshape(x_shape)  # reshape x back to its original shape
#     F = y[x_size:x_size+F_size]

#     differences = grounds - x[:, np.newaxis]
#     distances = np.linalg.norm(differences, axis=2, keepdims=True)

#     # ship_distances = np.linalg.norm(ship_diffences, axis=2)

#     # Calculate D matrix
#     D = differences / (distances + NORMALIZATOR)

#     # Calculate G matrix
#     G = ship_ground_gaussians(x, grounds, alpha, b=b)

#     # Calculate Collection matrix
#     C = ship_ground_gaussians(x, grounds, sigma, b=b)

#     # Calculate A matrix
#     A = attraction_matrix(U, G, D, F)

#     # Calculate H
#     H = harbor_attraction(U, x, h)


#     S = np.sum(A, axis=1).reshape(-1, 2)

#     S = S + H

#     S_normal = S / (np.linalg.norm(S, axis=1, keepdims=True) + NORMALIZATOR)

#     F_prime = calculate_F_prime(t, F, C, rate)

#     x_prime = calculate_x_prime(U, S_normal)

#     return np.concatenate((x_prime.flatten(), F_prime.flatten()))

# def costate(t, y, params):
#     x, F, alpha, sigma, beta, epsilon, rate, U, grounds, h, p_size, pf_size, p_shape = params
#     b = 1
#     p = y[:p_size].reshape(p_shape)  # reshape x back to its original shape
#     pf = y[p_size:p_size+pf_size]
#     x = x.reshape(p_shape)

#     differences = grounds - x[:, np.newaxis]
#     distances = np.linalg.norm(differences, axis=2, keepdims=True)

    
#     ship_diffences = x[:, np.newaxis] - x

#     # ship_distances = np.linalg.norm(ship_diffences, axis=2)

#     # Calculate D matrix
#     D = differences / (distances + NORMALIZATOR)

#     # Calculate G matrix
#     G = ship_ground_gaussians(x, grounds, alpha, b=b)

#     # Calculate Collection matrix
#     C = ship_ground_gaussians(x, grounds, sigma, b=b)

#     # Calculate A matrix
#     A = attraction_matrix(U, G, D, F)

#     # Calculate H
#     H = harbor_attraction(U, x, h)


#     S = np.sum(A, axis=1).reshape(-1, 2)

#     # S = S + H


#     Dxs = Dxs_matrix(x, grounds, x, G, U, F, distances, differences, alpha, h)

#     p_prime = calculate_p_prime(p, pf, S, U, Dxs, ship_diffences, differences, distances, rate, sigma, beta, epsilon)

#     pf_prime = calculate_p_f_prime(t, pf, C, rate)

#     return np.concatenate((p_prime.flatten(), pf_prime.flatten()))

# def solve_state_equations(state, params, t_final, y0, t_eval=1000):
#     """ Solves the state equations forward in time from t=0 to t=t_final. """
#     sol_state = solve_ivp(state, [0, t_final], y0, args=(params,), dense_output=True, t_eval=np.linspace(0, t_final, num=t_eval), method='RK45')
#     return sol_state

# def solve_costate_equations(costate, state_sol, params_costate, t_final, p0, x_size, F_size, t_eval=1000):
#     """ Solves the costate equations backward in time using interpolated state values. """
#     # Interpolate state solution
#     t_points = np.linspace(0, t_final, num=len(state_sol.t))
#     # state_interpolator = interp1d(t_points, state_sol.y.T, axis=0, kind='cubic', fill_value="extrapolate")

#     def costate_with_interpolated_state(t, y):
#         # Reverse the time for interpolation
#         x = state_sol.sol(t_final - t)[:x_size]
#         F = state_sol.sol(t_final - t)[x_size:x_size + F_size]
#         return costate(t, y, [x, F] + params_costate[2:])

#     # Solve costate from t_final to 0
#     sol_costate = solve_ivp(costate_with_interpolated_state, [t_final, 0], p0, t_eval=np.linspace(t_final, 0, num=t_eval), method='RK45')
#     return sol_costate

## Pontryagin forward–backward solver: explicit regularization and feedback

The active solver below uses forward states, backward adjoints, and joint Hamiltonian updates. The older helper definitions above remain for historical derivative tests; they are not the active solver. The route/timing controller is retained separately as a benchmark.

The new control coordinates are an effective target in the grounds’ convex hull and a speed fraction. Target regularization is `eta=0.5`; travel-cost smoothing is `nu=0.001`. Fish dynamics, physical attraction/speed bounds, and free terminal positions are preserved. Comparisons report the original travel cost. Convergence refers to the regularized formulation, not the unmodified raw-open-loop problem or a global optimum. See `audit/REPORT.md` for the equations and checks.

**Execution uses feedback:** `policy.controls(t, current_state)` reconstructs raw attractions and actual speed from current positions and stocks. The displayed `U_tensor` is a sample only. Freezing it into an open-loop table can produce large drift. This notebook uses the original dispersed starting positions; shared-harbor PMP optimization is not yet validated.

By default, the notebook loads an explicitly labeled, validated saved run. Set `RECOMPUTE=True` to perform a new optimization; this takes substantially longer than playback.


In [21]:
# # Constants and initial conditions
# alpha = 0.001
# sigma = 0.01
# beta = .1
# rate = 0.01
# delta = 100

# u_min = 0
# u_max = 10

# x_max, y_max = 200, 200
# n_ships = 20
# n_grounds = 30
# t_final = 500
# num_timesteps = 1000

# # Initialize scenario
# # grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
# # ships = generate_ships(n_ships, x_max, y_max)
# N = ships.shape[0]
# K = grounds.shape[0]
# h = np.array([x_max / 2, 0])

# # U and F initialization
# U = np.ones((N, K + 2))
# U[:, -2] = 0
# U[:, -1] = .5

# F0 = 1
# F = np.ones(K) * F0

# # Flatten initial conditions
# x = ships
# x_size = np.prod(x.shape)
# x_shape = x.shape
# F_size = np.prod(F.shape)
# p_size = x_size
# p_shape = x_shape
# y0 = np.concatenate((x.flatten(), F))

# p_init = np.zeros((n_ships, 2))
# pf_init = np.full(F.shape, -delta)
# p0 = np.concatenate((p_init.flatten(), pf_init))

# for i in range(10):

#     # Define parameters for state function
#     state_params = (alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape)
#     params_costate = [x, F, alpha, sigma, beta, rate, U, grounds, h, p_size, F_size, p_shape]

#     # Solve state equations
#     sol_state = solve_state_equations(state, state_params, t_final, y0, t_eval=num_timesteps)



#     print("done")
#     # Prepare initial conditions for costate solution


#     # Solve costate equations
#     sol_costate = solve_costate_equations(costate, sol_state, params_costate, t_final, p0, x_size, F_size, t_eval=num_timesteps)
    
#     # update initial conditions for p0
#     p0 = sol_costate.y[:, -1]
#     # Processing the results
#     # Reverse the solutions to match the original time order
#     costate_results = sol_costate.y[:, ::-1]

#     p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
#     pf_vectors = costate_results[p_size:]

#     x = sol_state.y[:x_size].reshape((n_ships, 2, -1))
#     F = sol_state.y[x_size:x_size+F_size]

#     for t in range(num_timesteps):
#         ship_positions = x[:, :, t]
#         fish_remaining = F[:, t]
#         D = d_matrix(ship_positions, grounds)
#         G = ship_ground_gaussians(ship_positions, grounds, alpha)
#         A = attraction_matrix(U, G, D, fish_remaining)
#         H = harbor_attraction(U, ship_positions, h)

#         S = calculate_S_matrix(A, H)
#         print(S.shape)
#         S = S / (np.linalg.norm(S, axis=1, keepdims=True) + NORMALIZATOR
#         )
        
#         for ship in range(n_ships):
#             # print(f"Ship {ship+1}")
#             p = p_vectors[ship, :, t]
#             # orthogonal = orthogonal_vectors[ship]
#             vision = G[ship, :] * F[ship, t]
#             # print(vision)
#             # Get the entries of vision contributing to 95% of the total vision
#             vision_threshold = np.percentile(vision, 95)
#             vision_mask = vision > vision_threshold

#             # Get the indices of the non-zero entries
#             indices = np.where(vision_mask)[0]
#             # print(indices)

#             # Get the coordinates of the fishing grounds that are not masked relative to the ship
#             control_grounds = grounds[indices] - ship_positions[ship]

#             # Get the fish remaining at the control grounds
#             control_fish = fish_remaining[indices]

#             # Get the controls from U for the control grounds
#             control_U = U[ship, indices]

#             # Get the vision of the ship on the control grounds
#             control_vision = vision[indices]

#             # Get the direction of the ship to the control grounds
#             control_direction = D[ship, indices]

#             print("TEST")
#             # print(control_U.shape, control_vision.shape, control_direction.shape, control_fish.shape)
#             s = S[ship]
#             # print(s.shape)
#             orthogonal = np.array([s[1], -s[0]])  # Assuming s is 2D
#             # print(orthogonal.shape)
#             # print(p.shape, orthogonal.shape)
#             dot_product = np.dot(p, orthogonal)
#             print(dot_product)

#             def minimize_dot_product(control_U):
#                 attraction = control_U * control_vision * control_direction * control_fish
#                 s = np.sum(attraction, axis=0)
#                 orthogonal = np.array([s[1], -s[0]])  # Assuming s is 2D
#                 dot_product = np.dot(p, orthogonal)
#                 return np.sum(dot_product**2)
            
#             res = minimize(minimize_dot_product, control_U, bounds=[(u_min, u_max)] * len(control_U))
#             # print(res)
#             U[ship, indices] = res.x


#             # control_U = U[ship, indices]
#             print("UPDATED")
#             A = attraction_matrix(U, G, D, fish_remaining)
#             H = harbor_attraction(U, ship_positions, h)
#             S = calculate_S_matrix(A, H)


#             # print(attraction.shape)
#             s = S[ship]
#             # print(s.shape)
#             orthogonal = np.array([s[1], -s[0]])  # Assuming s is 2D
#             # print(orthogonal.shape)
#             dot_product = np.dot(p, orthogonal)
#             print(dot_product)

#     print(f"Iteration {i+1} done")

# # print the control matrix
# # print(U)

# fig = plt.figure(figsize=(12, 6))
# fig.patch.set_facecolor('none')

# ax = fig.add_subplot(111)
# ax.patch.set_facecolor('white')

# plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# cbar = plt.colorbar()
# cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

# plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

# for i in range(x.shape[0]):
#     plt.plot(x[i, 0, :], x[i, 1, :])

# plt.xlabel('x')
# plt.ylabel('y')
# plt.title('State Trajectories')

# plt.xlim(0, x_max)
# # plt.ylim(0, y_max)

# fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
# starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
# trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

# plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# # plt.savefig('massive_resolution.png', transparent=True)
# plt.show()


# fig, ax = plt.subplots(figsize=(12, 6))
# ax.set_facecolor('white')

# # Plot fishing grounds
# plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# # Initial positions of ships
# plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^', label='Initial Boat Positions')

# # Choose a specific number of ships to plot
# num_ships_to_plot = 2
# # You can select ships randomly:
# # selected_ships = np.random.choice(range(n_ships), num_ships_to_plot, replace=False)
# # Or select specific ships by their indices:
# # selected_ships = [0, 1, 2]  # for example, to choose the first three ships

# # Plot trajectories for selected ships only
# for i in selected_ships:
#     plt.plot(x[i, 0, :], x[i, 1, :], label=f'Ship {i+1} Trajectory')

# # Normalize p vectors
# p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
# norms = np.linalg.norm(p_vectors[selected_ships], axis=1, keepdims=True)
# normalized_p_vectors = p_vectors[selected_ships] / (norms + NORMALIZATOR
# )

# # Adjust step for plotting arrows
# step = max(1, num_timesteps // 25)

# # Plot p vectors for selected ships
# for i, idx in enumerate(selected_ships):
#     indices = np.arange(0, num_timesteps, step)
#     # indices = indices[indices < num_timesteps]  # Ensure indices are within bounds
#     ax.quiver(x[idx, 0, indices], x[idx, 1, indices], normalized_p_vectors[i, 0, indices], normalized_p_vectors[i, 1, indices], color='red', scale=30, headwidth=2, headlength=2.5, width=0.005, headaxislength=2)

# # Set labels and titles
# plt.xlabel('x')
# plt.ylabel('y')
# plt.title('State Trajectories with Costate Vectors for Selected Ships')

# # Set plot limits
# plt.xlim(0, x_max)

# # Legend setup
# fishing_grounds = mlines.Line2D([], [], color='blue', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
# starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
# trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')
# plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.show()


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "audit").is_dir():
    root = root.parent
if not (root / "audit").is_dir():
    raise RuntimeError("Open this notebook from the repository root or random/ directory.")
if str(root / "audit") not in sys.path:
    sys.path.insert(0, str(root / "audit"))
from fishing import scenario
from regularized_pmp import RegularizedSweep
from pontryagin_controller import fit, save, load_saved, FeedbackPolicy
from pontryagin_visuals import sample_run

RECOMPUTE = False
INITIALIZATION = "coverage"  # unoptimized assignment; no route optimizer
RESULT_NAME = "coverage_400_verified"
# For T=400, RESULT_NAME="best_400" shows the benchmark-initialized PMP result.
problem = scenario(T=400.)
if RECOMPUTE:
    from finalize_pmp import finalize
    solver = RegularizedSweep(problem, steps=200, eta=.5, travel_smoothing=.001)
    control, solution = fit(solver, solver.seed(INITIALIZATION), cycles=6, iterations=800)
    candidate_name = "recomputed_400_candidate"
    save(candidate_name, solver, control, solution)
    solver, control, solution = finalize(
        root / "audit/results/pontryagin" / (candidate_name + ".npz"),
        candidate_name + "_checked", T=problem.T,
        initialization=INITIALIZATION + " initialization; fresh forward-backward optimization")
    playback_label = "New optimization"
else:
    solver, control, solution = load_saved(RESULT_NAME, problem)
    playback_label = "Saved validated result (no optimization rerun)"
policy = FeedbackPolicy(solver, control)
rollout = sample_run(solver, control)
rollout["objective"] = solution["original_J"]
verification = {key: solution[key] for key in [
    "original_equation_feedback_replays", "original_control_checks",
    "integrated_hamiltonian_gap", "max_hamiltonian_gap", "projected_residual"]}

N, K, F0 = problem.N, problem.K, problem.F0
grounds, ships = problem.grounds, problem.ships
t_final, time_grid = problem.T, rollout["times"]
num_timesteps = len(time_grid)
x = rollout["x"].transpose(1, 2, 0)
F = rollout["fish"].T
U_tensor = rollout["controls"]  # DISPLAY ONLY: execute policy.controls(t, current_state)
history = solution.get("history", [])
J_history = ([solution["initial_J"]] if solution.get("initial_J") is not None else []) + [row["J"] if isinstance(row, dict) else row for row in history]
if not J_history:
    J_history = [solution["J"]]
U_history, x_history = [U_tensor.copy()], [x.copy()]
stop_reason = solution["status"]
print(playback_label)
print("Initialization:", solution["initialization"])
print(f"Harvested {rollout['catch'][-1]:.9f} / {K*F0:g}")
print(f"Original-cost J* = {solution['original_J']:.9f}; regularized J = {solution['J']:.9f}")
print(stop_reason)
print(f"Projected residual {solution['projected_residual']:.3g}; integrated Hamiltonian gap {solution['integrated_hamiltonian_gap']:.3g}")
print("Execution: state-feedback attractions, original physical ODEs. No unregularized/global-optimum claim.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time_grid, rollout["catch"])
axes[0].set(xlabel="Time", ylabel="Fish units", title="PMP harvest", ylim=(0, K*F0*1.02))
axes[1].plot(J_history, marker="o", markersize=2)
axes[1].set(xlabel="Saved optimization iteration / cycle", ylabel="Regularized J*", title="Reported optimization history" if history else "Verified final objective (history unavailable)")
axes[1].ticklabel_format(axis="y", style="plain", useOffset=False)
for ax in axes:
    ax.grid(alpha=.2)
fig.tight_layout()
display(fig)
plt.close(fig)


In [ ]:
# Actual paths, verified by the coupled ODE (not an adjoint plot).
fig, ax = plt.subplots(figsize=(8, 7))
colors = plt.get_cmap("tab10")
stock = ax.scatter(*grounds.T, c=F[:, -1]/F0, cmap="Blues", vmin=0, vmax=1,
                   s=150, edgecolor="black")
for j, point in enumerate(grounds):
    ax.annotate(f"G{j+1}", point, (6, 6), textcoords="offset points")
for i in range(N):
    ax.plot(x[i, 0], x[i, 1], color=colors(i), label=f"Boat {i+1}")
    ax.scatter(*ships[i], marker="x", color=colors(i))
    ax.scatter(*x[i, :, -1], marker="^", color=colors(i))
ax.set(xlim=(-5, 205), ylim=(-5, 205), aspect="equal", xlabel="x", ylabel="y",
       title=f"PMP feedback policy: {rollout['catch'][-1]:.3f} / {K*F0:g} harvested")
ax.legend()
fig.colorbar(stock, ax=ax, label="Remaining / initial fish stock")
display(fig)
plt.close(fig)


In [ ]:
# Physical speed and dominant effective attraction, for display only.
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for i in range(N):
    axes[0].plot(time_grid, rollout["speed"][:, i], label=f"Boat {i+1}")
    dominant = (1 + np.argmax(rollout["effective_attractions"][:, i], axis=1)).astype(float)
    dominant[rollout["speed"][:, i] < 1e-4] = np.nan
    axes[1].step(time_grid, dominant, where="post")
axes[0].set(ylabel="Actual physical speed", ylim=(-.05, problem.V+.05))
axes[0].legend(ncol=5)
axes[1].set(xlabel="Time", ylabel="Largest attraction contribution", yticks=range(1, K+1))
fig.suptitle("Feedback control: up to two active ground attractions; heading is irrelevant at rest")
fig.tight_layout()
display(fig)
plt.close(fig)


Historical experiment (inactive; old solver API).

```python
x_max, y_max = 200, 200

# Constants and initial conditions
alpha = 0.0005
sigma = 0.05
beta = 0
rate = 0.01
delta = 100
gamma = 0.00001
epsilon = .001
V = 1

u_min = 0
u_max = 10

n_ships = 5
n_grounds = 7
t_final = 400
num_timesteps = 200

# Initialize scenario
# grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
# ships = generate_ships(n_ships, x_max, y_max)
N = ships.shape[0]
K = grounds.shape[0]
h = np.array([x_max / 2, 0])

# U and F initialization
U = np.ones((N, K + 2))
U[:, -2] = 0
U[:, -1] = 1

F0 = 100
F = np.ones(K) * F0

# Flatten initial conditions
x = ships
x_size = np.prod(x.shape)
x_shape = x.shape
F_size = np.prod(F.shape)
p_size = x_size
p_shape = x_shape
y0 = np.concatenate((x.flatten(), F))

p_init = np.zeros((n_ships, 2))
pf_init = np.full(F.shape, -delta)
p0 = np.concatenate((p_init.flatten(), pf_init))

for i in range(20):

    # Define parameters for state function
    state_params = (alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape)
    params_costate = [x, F, alpha, sigma, beta, epsilon, rate, U, grounds, h, p_size, F_size, p_shape]

    # Solve state equations
    sol_state = solve_state_equations(state, state_params, t_final, y0, t_eval=num_timesteps)



    print("done")
    # Prepare initial conditions for costate solution


    # Solve costate equations
    sol_costate = solve_costate_equations(costate, sol_state, params_costate, t_final, p0, x_size, F_size, t_eval=num_timesteps)
    
    # update initial conditions for p0
    # p0 = sol_costate.y[:, -1]
    # Processing the results
    # Reverse the solutions to match the original time order
    costate_results = sol_costate.y[:, ::-1]

    p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
    pf_vectors = costate_results[p_size:]

    x = sol_state.y[:x_size].reshape((n_ships, 2, -1))
    F = sol_state.y[x_size:x_size+F_size]


    print("Updating U")
    for t in range(num_timesteps):
        ship_positions = x[:, :, t]
        fish_remaining = F[:, t]
        D = d_matrix(ship_positions, grounds)
        G = ship_ground_gaussians(ship_positions, grounds, alpha)
        
        
        for ship in range(n_ships):
            U_ship = U[ship, :-2]
            p = p_vectors[ship, :, t]

            # attraction = U_ship[:-2, None] * G[ship, :, None] * D[ship] * fish_remaining[:, None]
            # s = np.sum(attraction, axis=0)
            # orthogonal = np.array([-s[1], s[0]])
            # print(orthogonal.shape)
            # dot_product = np.dot(p, orthogonal)
            # print("Previous dot product", dot_product)
            def residuals(U_ship):
                attraction = U_ship[:, None] * G[ship, :, None] * D[ship] * fish_remaining[:, None]
                s = np.sum(attraction, axis=0)
                orthogonal = np.array([-s[1], s[0]])
                dot_product = np.dot(p, orthogonal)
                # Return the residual to be minimized (dot_product should be zero)
                return [dot_product]  # Return as a list or array

            # Define bounds for each control in U_ship, assuming U_ship includes all controls for a ship
            lower_bounds = [u_min] * K
            upper_bounds = [u_max] * K

            # Solving the least squares problem with bounds
            res = least_squares(residuals, U[ship, :-2], bounds=(lower_bounds, upper_bounds), method='trf')

            # print(res.x)
            # # print(res)
            U[ship, :-2] = res.x
            # U_ship = U[ship]
            # p = p_vectors[ship, :, t]
            # attraction = U_ship[:-2, None] * G[ship, :, None] * D[ship] * fish_remaining[:, None]
            # s = np.sum(attraction, axis=0)
            # orthogonal = np.array([-s[1], s[0]])
            # dot_product = np.dot(p, orthogonal)
            # print("Updated dot product", dot_product)

            # attraction = U[ship][:-2, None] * G[ship, :, None] * D[ship] * fish_remaining[:, None]
            # s = np.sum(attraction, axis=0)

            # print((np.dot(p, (s / (np.linalg.norm(s) + NORMALIZATOR))) + gamma) > 0)

            # if (np.dot(p, (s / (np.linalg.norm(s) + NORMALIZATOR))) + gamma) > 0:
            #     U[ship, -1] = V
            # else:
            #     U[ship, -1] = 0
    print(F[:, -1].shape)
    # sum up over the last time step to get the total fish remaining
    initial_fish = np.sum(F[:, 0])
    total_fish_remaining = np.sum(F[:, -1])
    print(initial_fish)
    print(total_fish_remaining)

    # # Calculate the cost functional

    print(f"Iteration {i+1} done")


```


Historical experiment (inactive; old solver API).

```python

F_normalized = F[:, -1] / np.max(F[:, -1])

fig = plt.figure(figsize=(12, 6))
fig.patch.set_facecolor('none')

ax = fig.add_subplot(111)
ax.patch.set_facecolor('white')

plt.scatter(grounds[:, 0], grounds[:, 1], c=F_normalized, cmap='viridis', marker='x', s=100, label='Fishing Grounds')

cbar = plt.colorbar()
cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

for i in range(x.shape[0]):
    plt.plot(x[i, 0, :], x[i, 1, :])

plt.xlabel('x')
plt.ylabel('y')
plt.title(fr'Optimal Control State Trajectories, $\beta = {beta}$')

plt.xlim(0, x_max)
# plt.ylim(0, y_max)

fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.savefig('massive_resolution.png', transparent=True)
plt.show()


fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('none')


# Plot fishing grounds
plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# Initial positions of ships
plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^', label='Initial Boat Positions')

# Choose a specific number of ships to plot
# num_ships_to_plot = 2
# You can select ships randomly:
# selected_ships = np.random.choice(range(n_ships), num_ships_to_plot, replace=False)
# Or select specific ships by their indices:
selected_ships = [i for i in range(n_ships)]  # for example, to choose the first three ships
# selected_ships = [0, 1]

# Plot trajectories for selected ships only
for i in selected_ships:
    plt.plot(x[i, 0, :], x[i, 1, :], label=f'Ship {i+1} Trajectory')

# Normalize p vectors
p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
norms = np.linalg.norm(p_vectors[selected_ships], axis=1, keepdims=True)
normalized_p_vectors = p_vectors[selected_ships] / (norms + NORMALIZATOR)

# Adjust step for plotting arrows
step = max(1, num_timesteps // 25)

# Plot p vectors for selected ships
for i, idx in enumerate(selected_ships):
    indices = np.arange(0, num_timesteps, step)
    # indices = indices[indices < num_timesteps]  # Ensure indices are within bounds
    ax.quiver(x[idx, 0, indices], x[idx, 1, indices], normalized_p_vectors[i, 0, indices], normalized_p_vectors[i, 1, indices], color='red', scale=30, headwidth=2, headlength=2.5, width=0.005, headaxislength=2)

# Set labels and titles
plt.xlabel('x')
plt.ylabel('y')
plt.title(fr'Optimal Control State Trajectories with Costate Vectors, $\beta = {beta}$')

# Set plot limits
plt.xlim(0, x_max)

# Legend setup
fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')
plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

plt.show()

```


Historical experiment (inactive; old solver API).

```python

# U and F initialization
U = np.ones((N, K + 2))
U[:, -2] = 0
U[:, -1] = 1

F0 = 100
F = np.ones(K) * F0

# Flatten initial conditions
x = ships
x_size = np.prod(x.shape)
x_shape = x.shape
F_size = np.prod(F.shape)
p_size = x_size
p_shape = x_shape
y0 = np.concatenate((x.flatten(), F))

p_init = np.zeros((n_ships, 2))
pf_init = np.full(F.shape, -delta)
p0 = np.concatenate((p_init.flatten(), pf_init))

for i in range(1):

    # Define parameters for state function
    state_params = (alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape)
    params_costate = [x, F, alpha, sigma, beta, rate, U, grounds, h, p_size, F_size, p_shape]

    # Solve state equations
    sol_state = solve_state_equations(state, state_params, t_final, y0, t_eval=num_timesteps)



    print("done")
    # Prepare initial conditions for costate solution


    # Solve costate equations
    sol_costate = solve_costate_equations(costate, sol_state, params_costate, t_final, p0, x_size, F_size, t_eval=num_timesteps)
    
    # update initial conditions for p0
    # p0 = sol_costate.y[:, -1]
    # Processing the results
    # Reverse the solutions to match the original time order
    costate_results = sol_costate.y[:, ::-1]

    p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
    pf_vectors = costate_results[p_size:]

    x = sol_state.y[:x_size].reshape((n_ships, 2, -1))
    F = sol_state.y[x_size:x_size+F_size]
```


Historical experiment (inactive; old solver API).

```python
print(F[:, -1].shape)
# sum up over the last time step to get the total fish remaining
initial_fish = np.sum(F[:, 0])
total_fish_remaining = np.sum(F[:, -1])
print(initial_fish)
print(total_fish_remaining)
```


Historical experiment (inactive; old solver API).

```python

F_normalized = F[:, -1] / np.max(F[:, -1])

fig = plt.figure(figsize=(12, 6))
fig.patch.set_facecolor('none')

ax = fig.add_subplot(111)
ax.patch.set_facecolor('white')

plt.scatter(grounds[:, 0], grounds[:, 1], c=F_normalized, cmap='viridis', marker='x', s=100, label='Fishing Grounds')

cbar = plt.colorbar()
cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

for i in range(x.shape[0]):
    plt.plot(x[i, 0, :], x[i, 1, :])

plt.xlabel('x')
plt.ylabel('y')
plt.title(fr'Optimal Control State Trajectories, $\beta = {beta}$')

plt.xlim(0, x_max)
# plt.ylim(0, y_max)

fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.savefig('massive_resolution.png', transparent=True)
plt.show()


fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('none')


# Plot fishing grounds
plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# Initial positions of ships
plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^', label='Initial Boat Positions')

# Choose a specific number of ships to plot
# num_ships_to_plot = 2
# You can select ships randomly:
# selected_ships = np.random.choice(range(n_ships), num_ships_to_plot, replace=False)
# Or select specific ships by their indices:
selected_ships = [i for i in range(n_ships)]  # for example, to choose the first three ships
# selected_ships = [0, 1]

# Plot trajectories for selected ships only
for i in selected_ships:
    plt.plot(x[i, 0, :], x[i, 1, :], label=f'Ship {i+1} Trajectory')

# Normalize p vectors
p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
norms = np.linalg.norm(p_vectors[selected_ships], axis=1, keepdims=True)
normalized_p_vectors = p_vectors[selected_ships] / (norms + NORMALIZATOR)

# Adjust step for plotting arrows
step = max(1, num_timesteps // 25)

# Plot p vectors for selected ships
for i, idx in enumerate(selected_ships):
    indices = np.arange(0, num_timesteps, step)
    # indices = indices[indices < num_timesteps]  # Ensure indices are within bounds
    ax.quiver(x[idx, 0, indices], x[idx, 1, indices], normalized_p_vectors[i, 0, indices], normalized_p_vectors[i, 1, indices], color='red', scale=30, headwidth=2, headlength=2.5, width=0.005, headaxislength=2)

# Set labels and titles
plt.xlabel('x')
plt.ylabel('y')
plt.title(fr'Optimal Control State Trajectories with Costate Vectors, $\beta = {beta}$')

# Set plot limits
plt.xlim(0, x_max)

# Legend setup
fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')
plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

plt.show()

```


Historical experiment (inactive; old solver API).

```python

F_normalized = F[:, -1] / np.max(F[:, -1])

fig = plt.figure(figsize=(12, 6))
fig.patch.set_facecolor('none')

ax = fig.add_subplot(111)
ax.patch.set_facecolor('white')

plt.scatter(grounds[:, 0], grounds[:, 1], c=F_normalized, cmap='viridis', marker='x', s=100, label='Fishing Grounds')

cbar = plt.colorbar()
cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

for i in range(x.shape[0]):
    plt.plot(x[i, 0, :], x[i, 1, :])

plt.xlabel('x')
plt.ylabel('y')
plt.title('State Trajectories Without Optimal Control')

plt.xlim(0, x_max)
# plt.ylim(0, y_max)

fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.savefig('massive_resolution.png', transparent=True)
plt.show()


fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor('white')

# Plot fishing grounds
plt.scatter(grounds[:, 0], grounds[:, 1], c=F_normalized, cmap='viridis', marker='x', s=100, label='Fishing Grounds')

# Initial positions of ships
plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^', label='Initial Boat Positions')

# Choose a specific number of ships to plot
num_ships_to_plot = 2
# You can select ships randomly:
# selected_ships = np.random.choice(range(n_ships), num_ships_to_plot, replace=False)
# Or select specific ships by their indices:
selected_ships = [0, 1, 2]  # for example, to choose the first three ships

# Plot trajectories for selected ships only
for i in selected_ships:
    plt.plot(x[i, 0, :], x[i, 1, :], label=f'Ship {i+1} Trajectory')

# Normalize p vectors
p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
norms = np.linalg.norm(p_vectors[selected_ships], axis=1, keepdims=True)
normalized_p_vectors = p_vectors[selected_ships] / (norms + NORMALIZATOR)

# Adjust step for plotting arrows
step = max(1, num_timesteps // 25)

# Plot p vectors for selected ships
for i, idx in enumerate(selected_ships):
    indices = np.arange(0, num_timesteps, step)
    # indices = indices[indices < num_timesteps]  # Ensure indices are within bounds
    ax.quiver(x[idx, 0, indices], x[idx, 1, indices], normalized_p_vectors[i, 0, indices], normalized_p_vectors[i, 1, indices], color='red', scale=30, headwidth=2, headlength=2.5, width=0.005, headaxislength=2)

# Set labels and titles
plt.xlabel('x')
plt.ylabel('y')
plt.title('State Trajectories Without Optimal Control with Costate Vectors')

# Set plot limits
plt.xlim(0, x_max)

# Legend setup
fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')
plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

plt.show()

```


Historical experiment (inactive; old solver API).

```python
print(F[:, -1].shape)
# sum up over the last time step to get the total fish remaining
initial_fish = np.sum(F[:, 0])
total_fish_remaining = np.sum(F[:, -1])
print(initial_fish)
print(total_fish_remaining)
```


Historical experiment (inactive; old solver API).

```python
# # Constants and initial conditions
# alpha = 0.001
# sigma = 0.01
# beta = 1
# rate = 0.01
# delta = 100

# u_min = 0
# u_max = 10

# x_max, y_max = 200, 200
# n_ships = 20
# n_grounds = 30
# t_final = 500
# num_timesteps = 1000

# # Initialize scenario
# # grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
# # ships = generate_ships(n_ships, x_max, y_max)
# N = ships.shape[0]
# K = grounds.shape[0]
# h = np.array([x_max / 2, 0])

# # U and F initialization
# U = np.ones((N, K + 2))
# U[:, -2] = 0
# U[:, -1] = .5

# F0 = 1
# F = np.ones(K) * F0

# # Flatten initial conditions
# x = ships
# x_size = np.prod(x.shape)
# x_shape = x.shape
# F_size = np.prod(F.shape)
# p_size = x_size
# p_shape = x_shape
# y0 = np.concatenate((x.flatten(), F))

# p_init = np.zeros((n_ships, 2))
# pf_init = np.full(F.shape, -delta)
# p0 = np.concatenate((p_init.flatten(), pf_init))

# for i in range(10):

#     # Define parameters for state function
#     state_params = (alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape)
#     params_costate = [x, F, alpha, sigma, beta, rate, U, grounds, h, p_size, F_size, p_shape]

#     # Solve state equations
#     sol_state = solve_state_equations(state, state_params, t_final, y0, t_eval=num_timesteps)



#     print("done")
#     # Prepare initial conditions for costate solution


#     # Solve costate equations
#     sol_costate = solve_costate_equations(costate, sol_state, params_costate, t_final, p0, x_size, F_size, t_eval=num_timesteps)
    
#     # update initial conditions for p0
#     p0 = sol_costate.y[:, -1]
#     # Processing the results
#     # Reverse the solutions to match the original time order
#     costate_results = sol_costate.y[:, ::-1]

#     p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
#     pf_vectors = costate_results[p_size:]

#     x = sol_state.y[:x_size].reshape((n_ships, 2, -1))
#     F = sol_state.y[x_size:x_size+F_size]



    






# fig = plt.figure(figsize=(12, 6))
# fig.patch.set_facecolor('none')

# ax = fig.add_subplot(111)
# ax.patch.set_facecolor('white')

# plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# cbar = plt.colorbar()
# cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

# plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

# for i in range(x.shape[0]):
#     plt.plot(x[i, 0, :], x[i, 1, :])

# plt.xlabel('x')
# plt.ylabel('y')
# plt.title('State Trajectories')

# plt.xlim(0, x_max)
# # plt.ylim(0, y_max)

# fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
# starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
# trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

# plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# # plt.savefig('massive_resolution.png', transparent=True)
# plt.show()


# fig, ax = plt.subplots(figsize=(12, 6))
# ax.set_facecolor('white')

# # Plot fishing grounds
# plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

# # Initial positions of ships
# plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^', label='Initial Boat Positions')

# # Choose a specific number of ships to plot
# num_ships_to_plot = 2
# # You can select ships randomly:
# # selected_ships = np.random.choice(range(n_ships), num_ships_to_plot, replace=False)
# # Or select specific ships by their indices:
# # selected_ships = [0, 1, 2]  # for example, to choose the first three ships

# # Plot trajectories for selected ships only
# for i in selected_ships:
#     plt.plot(x[i, 0, :], x[i, 1, :], label=f'Ship {i+1} Trajectory')

# # Normalize p vectors
# p_vectors = costate_results[:p_size].reshape(n_ships, 2, -1)
# norms = np.linalg.norm(p_vectors[selected_ships], axis=1, keepdims=True)
# normalized_p_vectors = p_vectors[selected_ships] / (norms + NORMALIZATOR)

# # Adjust step for plotting arrows
# step = max(1, num_timesteps // 25)

# # Plot p vectors for selected ships
# for i, idx in enumerate(selected_ships):
#     indices = np.arange(0, num_timesteps, step)
#     # indices = indices[indices < num_timesteps]  # Ensure indices are within bounds
#     ax.quiver(x[idx, 0, indices], x[idx, 1, indices], normalized_p_vectors[i, 0, indices], normalized_p_vectors[i, 1, indices], color='red', scale=30, headwidth=2, headlength=2.5, width=0.005, headaxislength=2)

# # Set labels and titles
# plt.xlabel('x')
# plt.ylabel('y')
# plt.title('State Trajectories with Costate Vectors for Selected Ships')

# # Set plot limits
# plt.xlim(0, x_max)

# # Legend setup
# fishing_grounds = mlines.Line2D([], [], color='blue', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
# starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
# trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')
# plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.show()

```


Historical experiment (inactive; old solver API).

```python
# print(F[:, -1].shape)
# # sum up over the last time step to get the total fish remaining
# total_fish_remaining = np.sum(F[:, -1])
# print(total_fish_remaining)
```


Historical experiment (inactive; old solver API).

```python
# def solve_state_costate(U, grounds, ships, t_final, num_timesteps, params):
#     """ Solve state and costate equations, updating U iteratively """
#     x_size = np.prod(ships.shape)
#     x_shape = ships.shape
#     F_size = grounds.shape[0]
#     p_size = x_size
#     p_shape = ships.shape
#     y0_state = np.concatenate((ships.flatten(), np.ones(F_size)))  # Initial conditions for state
    
#     # Setting the initial conditions for costate (p)
#     p0_costate = np.zeros(x_size + F_size)
    
#     # Initialize historical U for convergence checking
#     U_old = np.zeros_like(U)
    
#     converged = False
#     iteration = 0
#     while not converged:
#         # Update parameters for each timestep
#         for t in range(num_timesteps):
#             # Current timestep controls
#             U_t = U[:, :, t]
#             params_state = (alpha, sigma, beta, rate, U_t, grounds, h, x_size, F_size, x_shape)
#             params_costate = [ships, np.ones(F_size), alpha, sigma, beta, rate, U_t, grounds, h, p_size, F_size, p_shape]

#             # Solve state equations
#             sol_state = solve_ivp(state, [0, t_final], y0_state, args=(params_state,), method='RK45', t_eval=[t_final])
#             state_results = sol_state.y[:, -1]  # Get the last timestep results
            
#             # Solve costate equations
#             sol_costate = solve_ivp(costate, [t_final, 0], p0_costate, args=(params_costate,), method='RK45', t_eval=[0])
#             costate_results = sol_costate.y[:, 0]  # Get the results at t=0
            

#         # Check convergence (based on some norm or criteria, e.g., difference in U)
#         U_change = np.linalg.norm(U - U_old)
#         if U_change < convergence_threshold or iteration > max_iterations:
#             converged = True
#         U_old = np.copy(U)
#         iteration += 1

#     return U  # Returns the updated U tensor

# # Constants and settings
# alpha = 0.001
# sigma = 0.01
# beta = 1
# rate = 0.01
# delta = 100
# u_min = 0
# u_max = 10
# x_max, y_max = 200, 200
# n_ships = 20
# n_grounds = 30
# t_final = 500
# num_timesteps = 1000
# convergence_threshold = 0.01
# max_iterations = 100
# h = np.array([x_max / 2, 0])  # Harbor position

# # Initialize scenarios and U tensor
# # grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
# # ships = generate_ships(n_ships, x_max, y_max)
# U = np.random.normal(1, 0.25, (n_ships, n_grounds + 2, num_timesteps))
# U[:, -2, :] = 0  # Example specific control adjustments
# U[:, -1, :] = 0.5

# # Solving the state and costate equations iteratively updating U
# U_final = solve_state_costate(U, grounds, ships, t_final, num_timesteps, None)
# print("Optimization completed.")

```


Historical experiment (inactive; old solver API).

```python
# # Constants and initialization
# alpha = 0.001
# sigma = 0.01
# beta = 1
# rate = 0.01
# delta = 100
# u_min = 0
# u_max = 10
# x_max, y_max = 200, 200
# n_ships = 20
# n_grounds = 30
# t_final = 500
# num_timesteps = 1000
# convergence_threshold = 0.01

# # Initialize U as a tensor of shape (n_ships, n_grounds + 2, num_timesteps)
# U = np.random.normal(1, .25, (n_ships, n_grounds + 2, num_timesteps))
# U[:, -2, :] = 0  # Specific control adjustments
# U[:, -1, :] = .5


# F0 = 1
# F = np.ones(K) * F0

# # Flatten initial conditions
# x = ships
# x_size = np.prod(x.shape)
# x_shape = x.shape
# F_size = np.prod(F.shape)
# p_size = x_size
# p_shape = x_shape
# y0 = np.concatenate((x.flatten(), F))

# def solve_equations(U_current):
#     # Define parameters for state function
#     state_params = (alpha, sigma, beta, rate, U_current, grounds, h, x_size, F_size, x_shape)
#     params_costate = [x, F, alpha, sigma, beta, rate, U_current, grounds, h, p_size, F_size, p_shape]

#     # Solve state equations
#     sol_state = solve_state_equations(state, state_params, t_final, y0, t_eval=num_timesteps)

#     # Prepare initial conditions for costate solution
#     p_init = np.zeros((n_ships, 2))
#     pf_init = np.full(F.shape, -delta)
#     p0 = np.concatenate((p_init.flatten(), pf_init))

#     # Solve costate equations
#     sol_costate = solve_costate_equations(costate, sol_state, params_costate, t_final, p0, x_size, F_size, t_eval=num_timesteps)

#     return sol_state, sol_costate

# converged = False
# prev_U = U.copy()
# iterations = 0
# while not converged:

#     state_results, costate_results = solve_equations(U)

#     x = state_results.y[:x_size].reshape((n_ships, 2, -1))
#     F = state_results.y[x_size:x_size+F_size]

#     p_vectors = costate_results.y[:p_size].reshape(n_ships, 2, -1)
#     pf_vectors = costate_results.y[p_size:]

#     for t in range(num_timesteps):
#         ship_positions = x[:, :, t]
#         fish_remaining = F[:, t]

#         differences = grounds - ship_positions[:, np.newaxis]

#         G = ship_ground_gaussians(ship_positions, grounds, alpha)

#         D = d_matrix(ship_positions, grounds)

#         for ship in range(n_ships):
#             p = p_vectors[ship, :, t]
#             vision = G[ship, :] * F[ship, t]

#             vision_threshold = np.percentile(vision, 95)
#             vision_mask = vision > vision_threshold

#             indices = np.where(vision_mask)[0]

#             control_grounds = grounds[indices] - ship_positions[ship]

#             control_fish = fish_remaining[indices]

#             control_U = U[ship, indices]

#             control_vision = vision[indices]

#             control_direction = D[ship, indices]

#             def minimize_dot_product(control_U):
#                 s = control_U * control_vision * control_direction * control_fish
#                 orthogonal = np.array([s[1], -s[0]])
#                 dot_product = np.dot(p, orthogonal)
#                 return np.sum(dot_product**2)
            
#             res = minimize(minimize_dot_product, control_U, bounds=[(u_min, u_max)] * len(control_U))
#             U[ship, indices] = res.x

#     # Check for convergence
#     diff = np.linalg.norm(U - prev_U)
#     if diff < convergence_threshold:
#         converged = True
#     prev_U = U.copy()
#     iterations += 1

#     print(f"Iteration {iterations}: Difference = {diff}")

# print("Converged!")
```


Historical experiment (inactive; old solver API).

```python
#OUTDATED
alpha = .001
sigma = .01
beta = 1
rate = .01

x_max, y_max = 200, 200
n_ships = 20
n_grounds = 30
t_final = 500

# h is the harbor position
h = np.array([x_max / 2, 0])

grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
ships = generate_ships(n_ships, x_max, y_max)
# print(ships.shape)
# ships = generate_ships_at_harbor(h, n_ships)

# put the grounds in the middle of both hemispheres

# put the ships at the harbor

# initialize U as all ones
U = np.ones((ships.shape[0], grounds.shape[0] + 2))

# initialize U with normal distribution, mean 1, std 0.1
U = np.random.normal(1, .25, (ships.shape[0], grounds.shape[0] + 2))
U = np.ones((ships.shape[0], grounds.shape[0] + 2))

# set the last column of U to 0
U[:,-2] = 0
U[:,-1] = .5

# initialize F as all ones
F = np.ones(grounds.shape[0])

# x is ship positions
x = ships



# Get the sizes of x and F before flattening
x_size = np.prod(x.shape)
x_shape = x.shape
F_size = np.prod(F.shape)

# Get the original shape of x

# Flatten x and F into a single 1D array
y0 = np.concatenate((x.flatten(), F))



state_params = (alpha, sigma, beta, rate, U, grounds, h, x_size, F_size, x_shape)

y_sol = solve_ivp(state, [0, t_final], y0, args=(state_params,), t_eval=np.linspace(0, t_final, t_final))

# Extract ship positions from the solution
x = y_sol.y[:x_size].reshape((n_ships, 2, -1))

# Extract F values from the solution
F = y_sol.y[x_size:x_size+F_size].T



# initialize p
p_init = np.full((n_ships, 2), 100)

# initialize pf, same size as F
pf_init = np.ones_like(F)
p_shape = p_init.shape
p_size = np.prod(p_init.shape)

# # Flatten p and pf into a single 1D array
# p0 = np.concatenate((p_init.flatten(), pf_init))

# costate_params = (x, F, 0, alpha, sigma, beta, rate, U, grounds, h, p_size, F_size, p_shape)

# p_sol = solve_ivp(costate, [0, t_final], p0, args=(costate_params,), t_eval=np.linspace(0, t_final, t_final))




fig = plt.figure(figsize=(12, 6))
fig.patch.set_facecolor('none')

ax = fig.add_subplot(111)
ax.patch.set_facecolor('white')

plt.scatter(grounds[:, 0], grounds[:, 1], c='blue', marker='x', s=100, label='Fishing Grounds')

cbar = plt.colorbar()
cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)

plt.scatter(x[:, 0, 0], x[:, 1, 0], c='green', marker='^')

for i in range(x.shape[0]):
    plt.plot(x[i, 0, :], x[i, 1, :])

plt.xlabel('x')
plt.ylabel('y')
plt.title('Empirical Evidence of Controlled Attraction')

plt.xlim(0, x_max)
# plt.ylim(0, y_max)

fishing_grounds = mlines.Line2D([], [], color='#fde725', marker='x', linestyle='None', markersize=10, label='Fishing Grounds')
starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Initial Boat Positions')
trajectories = mlines.Line2D([], [], color='blue', marker='_', linestyle='-', markersize=10, label='Trajectories')

plt.legend(handles=[fishing_grounds, starting_points, trajectories], loc='lower right')

# plt.savefig('massive_resolution.png', transparent=True)
plt.show()
```


Historical experiment (inactive; old solver API).

```python
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.set_xlim(0, 200)
# ax.set_ylim(0, 200)
# ax.set_xlabel('x')
# ax.set_ylabel('y')
# ax.set_title('Boat Trajectories')
# # Plot the fishing grounds and harbor for reference
# scat = ax.scatter(grounds[:, 0], grounds[:, 1], c=F_normalized, cmap='viridis', label='Fishing Grounds', marker='x', s=100)

# # Add colorbar
# cbar = plt.colorbar(scat)
# cbar.set_label('Percentage of Fish Remaining', rotation=270, labelpad=20)
# plt.tight_layout()

# # Initialize lines for each ship, storing them in a list
# lines = [ax.plot([], [], marker='^', linestyle='-', color='green', linewidth=0.5)[0] for _ in range(ships.shape[0])]

# def init():
#     # Initialize lines
#     for line in lines:
#         line.set_data([], [])
#     return lines

# def animate(frame):
#     # Update each ship's position for the current frame
#     for i, line in enumerate(lines):
#         line.set_data(ship_positions[i, 0, frame], ship_positions[i, 1, frame])
    
#     # Update the color of the fishing grounds
#     F_current = sol.y[:, frame][x_size:x_size+F_size]
#     F_normalized_current = (F_current / np.max(F)) * 100
#     scat.set_array(F_normalized_current)
    
#     return lines

# # Create the animation
# # Create the animation
# starting_points = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=10, label='Boat Positions')
# ani = FuncAnimation(fig, animate, frames=ship_positions.shape[2] - 1, init_func=init, blit=True, interval=20)

# plt.legend(handles=[fishing_grounds, starting_points], loc='upper left')
# plt.close(fig)

# # Display the animation
# HTML(ani.to_jshtml())
```


Historical experiment (inactive; old solver API).

```python
# # save the animation as gif
# ani.save('visualizations/boat_trajectories.gif', fps=30)
```


## Historical failed forward-backward repair (inactive)

This version accepted one update, then its adjoint failed. The working controller above supersedes it; the source remains here for audit provenance. It is not a successful controller.

```python
# Reproducible original setup; harbor attraction stays inactive.
np.random.seed(42)
x_max = y_max = 200
alpha, sigma, beta, rate = .005, .05, 5., .05
delta, gamma, epsilon, F0, V = 100., .01, .0001, 1., 1.
u_min, u_max = 0., 10.
n_ships, n_grounds = 5, 7
t_final, num_timesteps = 400, 200
grounds = generate_fishing_grounds(n_grounds, x_max, y_max)
ships = generate_ships(n_ships, x_max, y_max)
N, K = n_ships, n_grounds
h = np.array([x_max/2, 0.])
x_shape = p_shape = ships.shape
x_size = p_size = ships.size
F_size = K
y0 = np.r_[ships.ravel(), np.full(K,F0)]
p0 = np.r_[np.zeros(p_size), np.full(K,-delta)]
# A feasible initial guess that avoids hitting a ground center at full speed.
# This changes the initial guess only, not the dynamics or admissible controls.
U_tensor = np.zeros((num_timesteps,N,K+2))
time_grid = np.linspace(0,t_final,num_timesteps)
half_cell = (time_grid[1]-time_grid[0])/2
for ship in range(N):
    distances = np.linalg.norm(grounds-ships[ship],axis=1)
    target = int(np.argmin(distances))
    U_tensor[:,ship,target] = u_max
    travel_budget = max(0.,distances[target]-.5)/V
    U_tensor[:,ship,-1] = V*(time_grid+half_cell <= travel_budget)
# Column -2 is the inactive harbor control; column -1 is actual speed.
state_params = [alpha,sigma,beta,rate,grounds,h,x_size,F_size,x_shape,epsilon,gamma]
params_costate = [alpha,sigma,beta,epsilon,rate,gamma,grounds,h,p_size,F_size,p_shape]
U_history, P_history, pf_history, x_history, J_history = [], [], [], [], []
sol_state = solve_state_equations(state,U_tensor,state_params,t_final,y0,num_timesteps)
J_history.append(fishing_objective(sol_state,x_size,F_size,delta))
print(f'Initial J without self constant: {J_history[-1]:.9g}')
stop_reason = 'Iteration limit reached; convergence is not established.'
costate_failed = False
for iteration in range(10):
    try:
        sol_costate = solve_costate_equations(costate,sol_state,U_tensor,params_costate,
                                             t_final,p0,x_size,F_size,num_timesteps)
    except (RuntimeError, ValueError) as error:
        stop_reason = f"Numerical adjoint failure: {error}. No convergence claim."
        costate_failed = True
        break
    costate_results = sol_costate.y[:,::-1]
    p_vectors = costate_results[:p_size].reshape(N,2,-1)
    proposal = U_tensor.copy()
    for t in range(num_timesteps):
        positions = sol_state.y[:x_size,t].reshape(x_shape)
        fish = sol_state.y[x_size:x_size+F_size,t]
        D = d_matrix(positions,grounds)
        G = ship_ground_gaussians(positions,grounds,alpha)
        for ship in range(N):
            p = p_vectors[ship,:,t]
            basis = G[ship,:,None]*D[ship]*fish[:,None]
            scale = max(np.linalg.norm(p)+gamma, 1e-12)
            def negative_reward(weights):
                s = weights @ basis
                w = s/(np.linalg.norm(s)+NORMALIZATOR)
                return -(p@w-gamma*np.linalg.norm(w))/scale
            # Boundary controls are admissible. Test them as well as the current
            # weights; a zero stationarity residual is not an H maximum.
            starts = [U_tensor[t,ship,:K]] + list(u_max*np.eye(K))
            guess = min(starts,key=negative_reward)
            fit = minimize(negative_reward,guess,bounds=[(u_min,u_max)]*K,
                           method='L-BFGS-B',options={'ftol':1e-12,'maxiter':100})
            weights = fit.x if fit.success and fit.fun <= negative_reward(guess) else guess
            proposal[t,ship,:K] = weights
            proposal[t,ship,-1] = V if negative_reward(weights)<0 else 0.
    # Accept only a newly simulated decrease in the true J. Smoothing controls
    # by itself does not establish improvement, and speed must use column -1.
    old_J = J_history[-1]
    accepted = False
    for step in [1.,.5,.25,.125,.0625,.03125,.015625,.0078125]:
        trial_U = U_tensor+step*(proposal-U_tensor)
        try:
            trial_sol = solve_state_equations(state,trial_U,state_params,t_final,y0,num_timesteps)
        except (RuntimeError, ValueError):
            continue
        trial_J = fishing_objective(trial_sol,x_size,F_size,delta)
        if trial_J < old_J-1e-7*max(1.,abs(old_J)):
            U_tensor,sol_state = trial_U,trial_sol
            J_history.append(trial_J)
            accepted = True
            break
    if not accepted:
        stop_reason = 'No decreasing tested step; stopped without claiming optimality.'
        break
    caught = K*F0-sol_state.y[x_size:x_size+F_size,-1].sum()
    print(f'Update {iteration+1}: J={trial_J:.9g}, catch={caught:.9g}, step={step:g}')
print(stop_reason)
# Final plotted states and costates must correspond to the FINAL controls.
if costate_failed:
    costate_results = np.full((p_size+F_size,num_timesteps),np.nan)
else:
    try:
        sol_costate = solve_costate_equations(costate,sol_state,U_tensor,params_costate,
                                             t_final,p0,x_size,F_size,num_timesteps)
        costate_results = sol_costate.y[:,::-1]
    except (RuntimeError, ValueError) as error:
        print(f'Final costate unavailable: {error}')
        costate_results = np.full((p_size+F_size,num_timesteps),np.nan)
p_vectors = costate_results[:p_size].reshape(N,2,-1)
pf_vectors = costate_results[p_size:]
x = sol_state.y[:x_size].reshape(N,2,-1)
F = sol_state.y[x_size:x_size+F_size]
U_history.append(U_tensor.copy())
P_history.append(p_vectors.copy())
pf_history.append(pf_vectors.copy())
x_history.append(x.copy())
F_normalized = F[:,-1]/F0  # fraction of INITIAL stock, not final maximum
print(f'Final catch: {K*F0-F[:,-1].sum():.9g}; J={J_history[-1]:.9g}')
print(f'Self-crowding constant omitted from J above: {beta*N*t_final/epsilon:g}')
fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].scatter(*grounds.T,c=F_normalized,vmin=0,vmax=1,cmap='viridis',marker='x',s=80)
for ship in range(N):
    axes[0].plot(x[ship,0],x[ship,1])
axes[0].scatter(*ships.T,marker='^',color='black')
axes[0].set(title='Final accepted controls',xlabel='x',ylabel='y',aspect='equal')
axes[1].plot(J_history,marker='o')
axes[1].set(title='Actual objective after accepted updates',xlabel='Accepted update',ylabel='J minus constant')
plt.tight_layout()
plt.show()

```


## Retained route/timing benchmark entry point

This was the previous active controller. Its implementation and validated results remain in `audit/working_controller.py` and `audit/results/working/`.

```python
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "audit").is_dir():
    root = root.parent
if not (root / "audit").is_dir():
    raise RuntimeError("Open this notebook from the repository root or random/ directory.")
if str(root / "audit") not in sys.path:
    sys.path.insert(0, str(root / "audit"))
from fishing import scenario
from working_controller import optimize, validate, save
from controller_visuals import outcome_figure

# Original parameters and map. Numerical search settings do not change J.
harbor = False
problem = scenario(harbor=harbor, T=400.)
solution, trajectories = optimize(problem, standoff=.5, max_passes=6)
verification, rollout = validate(problem, solution, trajectories)
scenario_name = ("harbor_" if harbor else "notebook_") + str(int(problem.T))
save(scenario_name, problem, solution, trajectories, verification, rollout)

# Sampled controls below are for display. The executable control schedule uses
# exact segment switches in trajectories / the exported schedule CSV.
N, K, F0 = problem.N, problem.K, problem.F0
grounds, ships = problem.grounds, problem.ships
t_final, time_grid = problem.T, rollout["times"]
num_timesteps = len(time_grid)
x = rollout["x"].transpose(1, 2, 0)
F = rollout["fish"].T
U_tensor = rollout["controls"]
J_history = solution["history"]
U_history, x_history = [U_tensor.copy()], [x.copy()]
stop_reason = solution["status"]
print(f"Harvested {rollout['catch'][-1]:.9f} / {K*F0:g}")
print(f"J* = {rollout['objective']:.9f}; {stop_reason}")
print("Two ODE tolerance/step checks and separate quadrature passed.")
print("No global-optimality or Pontryagin-convergence claim.")
fig = outcome_figure(problem, solution, rollout)
display(fig)
plt.close(fig)

```
